In [1]:
from models import MLPModel, RNNModel, LSTMModel, BERTModel
import torch
from torch.utils.data import Dataset, DataLoader
from torch import nn
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
import bertviz

In [2]:
torch.manual_seed(67)

def seed_worker(worker_id): # Manual seeding init function for dataloaders
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)
g = torch.Generator()
g.manual_seed(42)

In [3]:
# One hot encoding
def expand_label(lab):
    n = len(lab)
    l=[]
    for i in range(n):
        temp = np.zeros((n,1))
        temp[int(lab[i])] = 1
        l.extend(temp)
    exp_labs = np.array(l, dtype=float).flatten()
    return exp_labs

# Splitting inputs from rank labels
def X_y_split(arr, seq_len):
    return arr[:,:seq_len], arr[:,seq_len:]

# Creating the Dataset class
class RankingDataset(Dataset):
    def __init__(self, X_train, y_train, device, enableEmbeds):
        super().__init__()
        self.nums = torch.tensor(X_train).to(device=device, dtype=torch.long if enableEmbeds else torch.float) # Numerical values
        self.labs = torch.tensor(y_train).to(device=device, dtype=torch.long) # Rank values

    def __len__(self):
        return len(self.nums)
    
    def __getitem__(self, idx):
        return (self.nums[idx], self.labs[idx])


def test(model, test_dataloader, seq_len):
    model.eval()
    loss_fn = nn.CrossEntropyLoss()
    s_error = 0
    s_matches = 0
    i=0
    with torch.no_grad():
        for nums, labs in test_dataloader:
            pred = model.forward(nums)
            pred = pred[0].view(-1, seq_len, seq_len) if isinstance(pred, tuple) else pred.view(-1, seq_len, seq_len) 
            labs = labs.long()   
            error = loss_fn(pred, labs).item()

            pred_ranks = torch.argmax(pred, dim=1)
            for seq, lab, pre in zip(nums,labs, pred_ranks):
                sample_true = lab.cpu().tolist()
                sample_pred = pre.cpu().tolist()

                no_matches = sum(1 for p, t in zip(sample_pred, sample_true) if p == t)
                i+=1
                s_error += error
                s_matches += no_matches
                if i%100==0 or no_matches==seq_len: 
                    print(f"{i}.Sequence--->{seq}\nPredic-->\t",sample_pred, "\nActual-->\t",sample_true, "---> Error=",round(s_error/i,3), "---> No. matches=",no_matches,'\n')
        print(f"Testing loss={s_error/i}\nAverage matches={s_matches/i}")

In [4]:
# Some variables
seq_len=10 # Permissible seq_lens : 4, 6, 8, 10
vocab_size=1000
emb_dim = 5 # set to None if you dont want embeddings to be used
               # Note. BERT model implementation has not been implemented with non-embedded inputs, please assign dmodel normally!
batch_size=128

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
enableNormalisation = 0       # 0 for none, 1 for z-norm, 2 for global min-max norm, 3 for sequence-wise min-max norm

if enableNormalisation:
    emb_dim = None

model_map = {
    "mlp":  MLPModel(vocab_size, seq_len, emb_dim, device=device),
    "rnn":  RNNModel(vocab_size, seq_len, emb_dim, num_layers=1, device=device),
    "lstm": LSTMModel(vocab_size, seq_len, emb_dim, num_layers=1, bidirectional=True, device=device),
    "bert": BERTModel(N=3, h=2, dmodel=10, dk=5, dv=5, vocab_size=vocab_size, seq_len=seq_len, device=device, viz=True), 
}                                                                                             # viz=True makes the model record the attention
                                                                                              # scores, for use with bertviz library!
        

load_model = "bert"

model = model_map[load_model]

In [5]:
#Loading Data
dat = pd.read_csv(f".\\Datasets\\{seq_len}_ranking_dataset.csv", dtype=int).to_numpy() # [10000 rows x 20 columns]


dat_dummy, dat_test = train_test_split(dat, test_size=0.1, random_state=42, shuffle=True)
dat_train, dat_val = train_test_split(dat_dummy, test_size=0.1/0.9, random_state=42, shuffle=True)

X_train, y_train = X_y_split(dat_train, seq_len)
X_val, y_val = X_y_split(dat_val, seq_len)
X_test, y_test = X_y_split(dat_test, seq_len)

# Normalisations
if enableNormalisation==1: # (Z-normalisation)
    X_mean, X_std = X_train.mean(axis=0), X_train.std(axis=0) 
    X_test = (X_test-X_mean)/X_std   
elif enableNormalisation==2: # (Global Min-Max normalisation)
    X_max, X_min = X_train.max(axis=0), X_train.min(axis=0) 
    X_test = (X_test-X_min)/(X_max-X_max)
elif enableNormalisation==3: # (Sequence-wise Min-Max normalisation)
    X_test = (X_test-X_test.min(axis=-1, keepdims=True))/(X_test.max(axis=-1, keepdims=True)-X_test.min(axis=-1, keepdims=True))

In [6]:
testing_data_loader = DataLoader(dataset=RankingDataset(X_test, y_test, device, enableEmbeds=emb_dim!=None),
                                 batch_size=batch_size, shuffle=True, worker_init_fn=seed_worker)
print("Starting! (Loaders initiated)\nFetching model parameters...")

Starting! (Loaders initiated)
Fetching model parameters...


In [7]:

# Loading best set of model weights
dest = f'.\\Saved Models\\{seq_len}_{model.tag}_model.pth' if not enableNormalisation else f'.\\Saved Models\\norm{enableNormalisation}_{seq_len}_{model.tag}_model.pth'
model.load_state_dict(torch.load(dest))
print("Fetched model parameters!")

Fetched model parameters!


In [8]:
# Testing model
print("\nTesting model!")
test(model, testing_data_loader, seq_len)
print("Testing done.")


Testing model!
1.Sequence--->tensor([198, 991, 386, 955, 319, 717, 753, 202, 173, 808], device='cuda:0')
Predic-->	 [1, 9, 4, 8, 3, 5, 6, 2, 0, 7] 
Actual-->	 [1, 9, 4, 8, 3, 5, 6, 2, 0, 7] ---> Error= 0.883 ---> No. matches= 10 

2.Sequence--->tensor([183, 966, 823, 312, 768, 615, 330, 697, 386, 514], device='cuda:0')
Predic-->	 [0, 9, 8, 1, 7, 5, 2, 6, 3, 4] 
Actual-->	 [0, 9, 8, 1, 7, 5, 2, 6, 3, 4] ---> Error= 0.883 ---> No. matches= 10 

9.Sequence--->tensor([473, 937, 324, 737, 310, 118, 497,  36, 867, 421], device='cuda:0')
Predic-->	 [5, 9, 3, 7, 2, 1, 6, 0, 8, 4] 
Actual-->	 [5, 9, 3, 7, 2, 1, 6, 0, 8, 4] ---> Error= 0.883 ---> No. matches= 10 

27.Sequence--->tensor([890, 973, 407, 901, 136, 299, 516, 278, 500, 436], device='cuda:0')
Predic-->	 [7, 9, 3, 8, 0, 2, 6, 1, 5, 4] 
Actual-->	 [7, 9, 3, 8, 0, 2, 6, 1, 5, 4] ---> Error= 0.883 ---> No. matches= 10 

64.Sequence--->tensor([379, 831, 447, 477,  56, 585, 133, 236, 369, 708], device='cuda:0')
Predic-->	 [4, 9, 5, 6, 0, 7

In [9]:

print("Enter sequence for testing: ", end='') # Enter in format: x1,x2,x3,x4...,xn etc
seq = list(eval(input()))
if len(seq)>seq_len:
    seq = seq[:seq_len]
seq_tensor = torch.tensor([seq], dtype=torch.long, device=device)
if load_model == 'bert':
    pred, att = model.forward(seq_tensor)
    pred_rank = pred.view(-1, seq_len, seq_len).argmax(dim=1)
    print("Prediction:",pred_rank.tolist())
    attention = [layer_attn[0:1].cpu() for layer_attn in model.scores]
    bertviz.head_view(attention, [str(i) for i in seq]) # Pre-rendered bertviz is for seq = [511, 896,  95, 259, 304, 440, 782, 708, 156, 661]
else:
    pred = model.forward(seq_tensor)
    pred_rank = pred.view(-1, seq_len, seq_len).argmax(dim=1)
    print("Prediction:",pred_rank.tolist())


Enter sequence for testing: Prediction: [[5, 9, 0, 2, 3, 4, 8, 7, 1, 6]]


<IPython.core.display.Javascript object>